# Best/worst case analysis of the PGChecker runs

This notebook condenses **several PGChecker runs** into the overview that Hunold et al.
show in *Parallel Computing 129 (2026) 103205*, **Fig. 5 (page 7)**: one colour-coded
cell per collective and message size, once for the **best** and once for the **worst**
observed overhead.

## What gets pooled

The only input is a **substring of the `experiment_id`** (e.g. `pgchecker-4nodes`).
The notebook then collects every matching experiment under `results/`, and inside each
one **every allocation `k` with all of its repetitions `i`**. Each `(experiment, k, i)`
counts as one *execution* -- the counterpart of a batch job / node allocation in the
paper (there: three batch jobs per machine).

## The metric (paper, section 5.1.4)

Per execution, collective and message size:

* `eta_L` = median runtime of the **MPI library implementation** (`default`),
* `eta_M_j` = median runtime of the *j*-th **mock-up** / algorithmic variant,
* `eta_M = min_j eta_M_j` (fastest competitor, Eq. 10),
* `o = max(0, eta_L / eta_M - 1)` (overhead, Eq. 11).

`o` is reported as a percentage: `o = 0.3` means the library function was 30 % slower
than the fastest competitor. It is therefore **not** the same as the `slowdown` in
PGChecker's output, which is the plain ratio of the two medians.

The medians are recomputed straight from the `.dat` raw data (median over all `nrep`
rows per `count`). Section 4 checks that against PGChecker's own
`grouped_violation/*.txt` output.

In [ ]:
from pathlib import Path
import json
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter, NullFormatter


def find_repo_root(start: Path) -> Path:
    # Walk upwards so the notebook also runs from a subdirectory, not just the repo root.
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "results").is_dir():
            return candidate
    raise FileNotFoundError(f"No results/ directory above {start.resolve()}")


REPO_ROOT = find_repo_root(Path.cwd())
RESULTS_ROOT = REPO_ROOT / "results"

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
# Matched as a substring against the experiment_id (= directory name under
# results/); every matching experiment is pooled, with all of its k and i.
EXPERIMENT_FILTER = "pgchecker-uc-4nodes-3times-all"

# One of "mean" | "median" | "max" of o.
RANK_METRIC = "mean"

# Runs with a non-zero exit_code sometimes still carry usable raw data.
INCLUDE_FAILED_RUNS = False

SAVE_OUTPUT = True
OUTPUT_DIR = RESULTS_ROOT / "analysis" / EXPERIMENT_FILTER

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    # Fig. 5 of the paper is a LaTeX figure: serif text, black axes.
    "font.family": "serif",
    "mathtext.fontset": "dejavuserif",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelcolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "axes.edgecolor": "black",
    "savefig.facecolor": "white",
})

print("Repo root:", REPO_ROOT)
print("Results:  ", RESULTS_ROOT)
print("Filter:   ", EXPERIMENT_FILTER)

## Overhead categories and colours

Four categories as in the paper (section 5.1.4): `low` for `o <= 0.1`, `medium` for
`0.1 < o <= 0.5`, `high` for `0.5 < o <= 2.0`, `very high` for `o > 2.0`.
**Green is the `low` category** and includes `o = 0` -- there is no separate zero
category.

The paper's plotting code is not published ("Data will be made available on request").
Fill colours, cell shape and line width were therefore read straight out of the vector
paths of Fig. 5 in the PDF; the cell labels are black and carry no percent sign, just
like the original.

In [ ]:
# Read out of the vector paths of Fig. 5 in the PDF: the exact fill colors, the
# cell shape (23.2 x 15.9 pt) and the border width (0.61 pt). Green is the "low"
# category and covers o = 0, so there is no separate zero category.
SEVERITY = [
    ("($\leq$ 10%) low",      "#68b69c"),
    ("($\leq$ 50%) medium",   "#eea8a8"),
    ("($\leq$ 200%) high",    "#d46464"),
    ("($>$ 200%) very high",  "#ca3434"),
]
SEVERITY_LABELS = [row[0] for row in SEVERITY]
SEVERITY_COLORS = [row[1] for row in SEVERITY]

MISSING_COLOR = "#ffffff"   # cell without data
CELL_LINE_COLOR = "#000000"
CELL_LINE_WIDTH = 0.61
CELL_ASPECT = 15.9 / 23.2   # cells are wider than tall, as in the paper
CELL_WIDTH_IN = 0.72        # scaled up from the paper's 23.2 pt for screen use
THIN_SPACE = " "       # group separator in the axis labels, as in the paper


def severity_index(overhead: float) -> int:
    """Color category of an overhead value; -1 means no data."""
    if not np.isfinite(overhead):
        return -1
    if overhead <= 0.1:
        return 0
    if overhead <= 0.5:
        return 1
    if overhead <= 2.0:
        return 2
    return 3


def format_size(num_bytes: float) -> str:
    """80000 -> '80 000' (thin space as the group separator, as in the paper)."""
    return f"{int(num_bytes):,}".replace(",", THIN_SPACE)


def format_overhead(overhead: float) -> str:
    """Overhead as a rounded integer percentage, without a percent sign."""
    return "" if not np.isfinite(overhead) else f"{overhead * 100.0:.0f}"

## 1. Find the executions

Result layout as documented in `CLAUDE.md`: `results/{experiment_id}/k{k}/i{i}/` holding
`run.json`, `run.log`, the `.dat` raw data, `pgmpi_info.csv` and -- if PGChecker got
that far -- `grouped_violation/*.txt`.

In [ ]:
def load_json(path: Path) -> dict:
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (FileNotFoundError, json.JSONDecodeError):
        return {}


def discover_executions(filter_string: str, results_root: Path = RESULTS_ROOT) -> pd.DataFrame:
    """Every (experiment, k, i) run whose experiment_id contains `filter_string`."""
    experiment_dirs = sorted(
        path for path in results_root.iterdir()
        if path.is_dir() and filter_string in path.name
    )
    if not experiment_dirs:
        available = [path.name for path in sorted(results_root.iterdir()) if path.is_dir()]
        raise FileNotFoundError(
            f"No experiment in {results_root} contains '{filter_string}'. Available: {available}"
        )

    rows = []
    for exp_index, exp_dir in enumerate(experiment_dirs):
        experiment = load_json(exp_dir / "experiment.json")
        exp_tag = chr(ord("A") + exp_index) if exp_index < 26 else f"E{exp_index}"
        for k_dir in sorted(path for path in exp_dir.glob("k*") if path.is_dir()):
            allocation = load_json(k_dir / "allocation.json")
            machines = [node.get("name") for node in allocation.get("selected_nodes", [])
                        if node.get("name")]
            for i_dir in sorted(path for path in k_dir.glob("i*") if path.is_dir()):
                run = load_json(i_dir / "run.json")
                rows.append({
                    "execution": f"{exp_dir.name}/{k_dir.name}/{i_dir.name}",
                    "label": f"{exp_tag} {k_dir.name}/{i_dir.name}",
                    "experiment_id": exp_dir.name,
                    "experiment_name": experiment.get("name"),
                    "k": k_dir.name[1:],
                    "i": i_dir.name[1:],
                    "path": i_dir,
                    "exit_code": run.get("exit_code"),
                    "node_count": run.get("node_count"),
                    "processes_per_node": run.get("processes_per_node"),
                    "started_at": run.get("started_at"),
                    "duration_s": run.get("duration_seconds"),
                    "reservation_id": run.get("reservation_id") or allocation.get("reservation_id"),
                    "machines": ", ".join(machines),
                    "dat_files": len(list(i_dir.glob("data_*.dat"))),
                })

    found = pd.DataFrame(rows)
    found["usable"] = found["dat_files"] > 0
    if not INCLUDE_FAILED_RUNS:
        found["usable"] &= found["exit_code"].fillna(-1).eq(0)
    return found


found = discover_executions(EXPERIMENT_FILTER)
executions = found[found["usable"]].reset_index(drop=True)

skipped = found[~found["usable"]]
if not skipped.empty:
    print(f"Skipped ({len(skipped)} of {len(found)}) -- no raw data or exit_code != 0:")
    for row in skipped.itertuples(index=False):
        print(f"  {row.execution}  exit_code={row.exit_code}  dat_files={row.dat_files}")

if executions.empty:
    raise RuntimeError(f"No usable execution for filter '{EXPERIMENT_FILTER}'.")

print(f"\nUsable executions: {len(executions)}")
display(executions[[
    "label", "experiment_id", "k", "i", "node_count", "processes_per_node",
    "duration_s", "dat_files", "machines",
]])

## 2. Median runtimes per collective, message size and algorithm

The file names are `data_<MPI_Call>_<algname>.dat`. Since MPI names contain underscores
themselves (`MPI_Reduce_scatter_block`), the split is read from `pgmpi_info.csv` instead
of guessed; a name-based fallback only kicks in when that file is missing.

In [ ]:
DAT_COLUMNS = ["test", "nrep", "count", "runtime_sec"]

# Fallback for a missing pgmpi_info.csv: longest match first, so that
# MPI_Reduce_scatter_block is not read as MPI_Reduce.
KNOWN_COLLECTIVES = sorted([
    "MPI_Allgather", "MPI_Allreduce", "MPI_Alltoall", "MPI_Bcast", "MPI_Gather",
    "MPI_Reduce", "MPI_Reduce_scatter_block", "MPI_Scan", "MPI_Scatter",
], key=len, reverse=True)


def algorithm_files(exec_dir: Path) -> dict:
    """{(collective, algorithm): path} for every .dat file of one execution."""
    mapping = {}
    info_path = exec_dir / "pgmpi_info.csv"
    if info_path.exists():
        info = pd.read_csv(info_path, sep=";", comment="#")
        for collective, algorithm in info[["mpiname", "algname"]].itertuples(index=False):
            path = exec_dir / f"data_{collective}_{algorithm}.dat"
            if path.exists():
                mapping[(collective, algorithm)] = path

    mapped = set(mapping.values())
    for path in sorted(exec_dir.glob("data_*.dat")):
        if path in mapped:
            continue
        stem = path.stem[len("data_"):]
        collective = next((name for name in KNOWN_COLLECTIVES if stem.startswith(name + "_")), None)
        if collective is None:
            print(f"  Unknown file naming, skipped: {path.name}")
            continue
        mapping[(collective, stem[len(collective) + 1:])] = path
    return mapping


def parse_dat(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path, sep=r"\s+", comment="#")
    missing = [column for column in DAT_COLUMNS if column not in frame.columns]
    if missing:
        raise ValueError(f"{path.name}: missing columns {missing} (found: {list(frame.columns)})")
    return frame


def collect_medians(executions: pd.DataFrame) -> pd.DataFrame:
    """Median/mean runtime per (execution, collective, message size, algorithm)."""
    chunks = []
    for row in executions.itertuples(index=False):
        files = algorithm_files(row.path)
        if not files:
            print(f"  {row.label}: no .dat files -- skipped")
            continue
        for (collective, algorithm), path in sorted(files.items()):
            aggregated = (
                parse_dat(path)
                .groupby("count")["runtime_sec"]
                .agg(median_s="median", mean_s="mean", min_s="min", max_s="max", samples="count")
                .reset_index()
                .rename(columns={"count": "message_size"})
            )
            aggregated["execution"] = row.execution
            aggregated["label"] = row.label
            aggregated["collective"] = collective
            aggregated["algorithm"] = algorithm
            chunks.append(aggregated)
        print(f"  {row.label}: read {len(files)} algorithm files")

    medians = pd.concat(chunks, ignore_index=True)
    # PGChecker reports milliseconds -- same unit as grouped_violation.
    for column in ("median", "mean", "min", "max"):
        medians[f"{column}_ms"] = medians[f"{column}_s"] * 1000.0
    medians["is_default"] = medians["algorithm"] == "default"
    return medians


print("Reading raw data:")
medians = collect_medians(executions)

print(f"\nRows: {len(medians)}"
      f"\nCollectives:   {sorted(medians['collective'].unique())}"
      f"\nMessage sizes: {sorted(medians['message_size'].unique())}"
      f"\nAlgorithms:    {sorted(medians['algorithm'].unique())}")

## 3. Compute the overhead `o`

Per execution, collective and message size: the fastest mock-up (`eta_M`), its ratio to
the library implementation (`eta_L / eta_M`), and from that
`o = max(0, eta_L/eta_M - 1)`.

In [ ]:
CELL_KEYS = ["collective", "message_size"]
FULL_KEYS = ["execution", *CELL_KEYS]


def compute_overhead(medians: pd.DataFrame) -> pd.DataFrame:
    library = medians[medians["is_default"]]
    mockups = medians[~medians["is_default"]]

    duplicated = int(library.duplicated(FULL_KEYS).sum())
    if duplicated:
        raise ValueError(f"{duplicated} duplicate default measurements -- check the data layout")

    fastest = mockups.loc[mockups.groupby(FULL_KEYS)["median_ms"].idxmin()]
    mockup_count = mockups.groupby(FULL_KEYS)["algorithm"].nunique().rename("mockup_count")

    overhead = (
        library[[*FULL_KEYS, "label", "median_ms", "samples"]]
        .rename(columns={"median_ms": "eta_lib_ms", "samples": "nrep"})
        .merge(
            fastest[[*FULL_KEYS, "median_ms", "algorithm"]]
            .rename(columns={"median_ms": "eta_mock_ms", "algorithm": "fastest_mockup"}),
            on=FULL_KEYS, how="left",
        )
        .merge(mockup_count.reset_index(), on=FULL_KEYS, how="left")
    )
    overhead["slowdown"] = overhead["eta_lib_ms"] / overhead["eta_mock_ms"]
    overhead["overhead"] = (overhead["slowdown"] - 1.0).clip(lower=0.0)
    overhead["overhead_pct"] = overhead["overhead"] * 100.0
    overhead["severity"] = overhead["overhead"].map(severity_index)
    return overhead.sort_values(["collective", "message_size", "execution"]).reset_index(drop=True)


overhead = compute_overhead(medians)

COLLECTIVES = sorted(overhead["collective"].unique())
MESSAGE_SIZES = sorted(overhead["message_size"].unique())
print(f"{len(overhead)} cells = {len(COLLECTIVES)} collective(s) x {len(MESSAGE_SIZES)} "
      f"size(s) x {overhead['execution'].nunique()} executions")
print(f"Mock-ups compared per cell: {sorted(overhead['mockup_count'].unique())}")

print("\nOverhead o [%] per cell and execution:")
display(
    overhead.pivot_table(index=["collective", "message_size"], columns="label",
                        values="overhead_pct").round(1)
)

## 4. Cross-check against PGChecker's own output

`grouped_violation/<MPI_Call>.txt` carries `default_median`, `slowdown` and
`fastest_mockup` as computed by PGChecker itself -- the deviation from the values
recomputed here has to be about zero. Rows without a `slowdown` are the cases where no
mock-up was faster; there `o = 0`.

In [ ]:
VIOLATION_COLUMNS = [
    "collective", "count", "N", "ppn", "n", "default_median", "default_mean",
    "slowdown", "stat_violation", "fastest_mockup", "mockup_median", "mockup_mean",
]


def parse_grouped_violation(path: Path) -> pd.DataFrame:
    """Read PGChecker's own summary; the columns are fixed, missing fields become NaN."""
    frame = pd.read_csv(path, sep=r"\s+", skiprows=2, names=VIOLATION_COLUMNS,
                       header=None, engine="python")
    return frame.rename(columns={"count": "message_size"})


checks = []
for row in executions.itertuples(index=False):
    for path in sorted((row.path / "grouped_violation").glob("*.txt")):
        frame = parse_grouped_violation(path)
        frame["execution"] = row.execution
        checks.append(frame)

if checks:
    reported = pd.concat(checks, ignore_index=True)
    comparison = overhead.merge(
        reported[["execution", "collective", "message_size", "default_median",
                  "slowdown", "fastest_mockup", "stat_violation"]],
        on=FULL_KEYS, how="left", suffixes=("", "_pgchecker"),
    )
    reported_slowdown = pd.to_numeric(comparison["slowdown_pgchecker"], errors="coerce")
    comparison["d_median_ms"] = (comparison["eta_lib_ms"] - comparison["default_median"]).abs()
    comparison["d_slowdown"] = (comparison["slowdown"] - reported_slowdown).abs()
    # PGChecker only names a fastest_mockup when some mockup actually was faster.
    same_mockup = (
        comparison["fastest_mockup_pgchecker"].isna()
        | (comparison["fastest_mockup_pgchecker"]
           == comparison["collective"] + "_" + comparison["fastest_mockup"])
        | (comparison["fastest_mockup_pgchecker"] == comparison["fastest_mockup"])
    )
    print(f"Cells with a PGChecker reference: {int(reported_slowdown.notna().sum())}")
    print(f"max |median deviation| [ms]:      {comparison['d_median_ms'].max():.3g}")
    print(f"max |slowdown deviation|:         {comparison['d_slowdown'].max():.3g}")
    print(f"fastest_mockup identical:         {bool(same_mockup.all())}")
    display(
        comparison.loc[
            reported_slowdown.notna(),
            ["label", "collective", "message_size", "eta_lib_ms", "default_median",
             "slowdown", "slowdown_pgchecker", "fastest_mockup",
             "fastest_mockup_pgchecker", "stat_violation"],
        ].head(12)
    )
else:
    print("No grouped_violation/*.txt found -- cross-check skipped.")

## 5. Best and worst execution

Two readings, both of which the paper uses:

1. **per cell** (the default, *"best and worst observed overheads across the batch
   jobs"*): for each collective and size separately, the minimum resp. maximum over all
   executions. The best and the worst panel may therefore come from different runs.
2. **per execution** (the caption of Fig. 5, *"the best and the worst execution"*): the
   execution with the lowest resp. highest overhead overall (metric `RANK_METRIC`), each
   shown in full.

In [ ]:
def extreme_cells(overhead: pd.DataFrame, mode: str) -> pd.DataFrame:
    """Per (collective, message size) the execution with the lowest/highest overhead.

    `source` names that execution, or -- when several reach the same extreme,
    which is the common case for o = 0 -- how many of them did.
    """
    grouped = overhead.groupby(CELL_KEYS)["overhead"]
    picks = overhead.loc[grouped.idxmin() if mode == "min" else grouped.idxmax()].copy()

    extremes = picks.set_index(CELL_KEYS)["overhead"]
    per_row = extremes.reindex(pd.MultiIndex.from_frame(overhead[CELL_KEYS])).to_numpy()
    tied = np.isclose(overhead["overhead"].to_numpy(), per_row, rtol=1e-9, atol=0.0)
    ties = overhead[tied].groupby(CELL_KEYS)["execution"].nunique()

    picks["ties"] = ties.reindex(pd.MultiIndex.from_frame(picks[CELL_KEYS])).to_numpy()
    picks["source"] = np.where(picks["ties"] > 1,
                               picks["ties"].astype(str) + " runs", picks["label"])
    return picks


best_cells = extreme_cells(overhead, "min")
worst_cells = extreme_cells(overhead, "max")

ranking = (
    overhead.groupby(["execution", "label"])["overhead"]
    .agg(mean="mean", median="median", max="max", cells="count")
    .reset_index()
    .sort_values(RANK_METRIC)
    .reset_index(drop=True)
)
for column in ("mean", "median", "max"):
    ranking[f"{column}_pct"] = (ranking[column] * 100).round(1)

best_execution = ranking.iloc[0]
worst_execution = ranking.iloc[-1]

print(f"Executions ranked by the '{RANK_METRIC}' of the overhead o:")
display(ranking[["label", "execution", "cells", "mean_pct", "median_pct", "max_pct"]])
print(f"Best execution:  {best_execution['label']}  ({best_execution['execution']})")
print(f"Worst execution: {worst_execution['label']}  ({worst_execution['execution']})")

spread = (
    worst_cells.set_index(CELL_KEYS)["overhead_pct"]
    - best_cells.set_index(CELL_KEYS)["overhead_pct"]
).rename("spread_pct").reset_index()
print("\nLargest spread between the best and the worst observation:")
display(spread.sort_values("spread_pct", ascending=False).head(5).round(1))

## 6. Overview heatmap (Fig. 5)

Left the best, right the worst observed overhead. The number in the cell is `o` as a
percentage. The paper prints no run label inside the cells, so the table below the figure
records where each extreme came from; when several executions reach the same extreme --
typical for 0 % -- it states how many did.

In [ ]:
def overhead_matrix(frame: pd.DataFrame, rows, message_sizes, column, row_key="collective"):
    """Rows = `rows` (collectives or executions), columns = message sizes."""
    return (
        frame.pivot(index=row_key, columns="message_size", values=column)
        .reindex(index=rows, columns=message_sizes)
    )


def plot_overhead_panels(panels, caption=None, filename=None,
                         rows=None, row_key="collective"):
    """panels: [(panel title, frame with one row per (row_key, message_size))]."""
    collectives = COLLECTIVES if rows is None else rows
    message_sizes = MESSAGE_SIZES
    cmap = ListedColormap(SEVERITY_COLORS)
    cmap.set_bad(MISSING_COLOR)

    # A fixed cell size keeps the paper's cell shape no matter how many rows
    # there are; with aspect="auto" a single collective would be stretched into
    # a bar across the whole panel width.
    width = CELL_WIDTH_IN * len(message_sizes) + 2.6
    height = CELL_WIDTH_IN * CELL_ASPECT * len(collectives) + 2.2
    fig, axes = plt.subplots(1, len(panels), figsize=(width * len(panels), height),
                             squeeze=False, layout="constrained")

    for ax, (panel_title, frame) in zip(axes[0], panels):
        values = overhead_matrix(frame, collectives, message_sizes,
                                 "overhead", row_key).to_numpy(float)
        severities = np.vectorize(severity_index)(values).astype(float)
        masked = np.ma.masked_where(severities < 0, severities)

        ax.imshow(masked, cmap=cmap, vmin=-0.5, vmax=len(SEVERITY_COLORS) - 0.5,
                  aspect=CELL_ASPECT)
        ax.set_title(panel_title, pad=10)
        ax.set_xticks(range(len(message_sizes)),
                      [format_size(size) for size in message_sizes], rotation=45, ha="right")
        ax.set_yticks(range(len(collectives)), collectives, rotation=45, ha="right",
                      fontfamily="monospace")
        # The minor grid draws the cell borders, the spines close the frame.
        ax.set_xticks(np.arange(-0.5, len(message_sizes), 1), minor=True)
        ax.set_yticks(np.arange(-0.5, len(collectives), 1), minor=True)
        ax.grid(which="minor", color=CELL_LINE_COLOR, linewidth=CELL_LINE_WIDTH)
        ax.tick_params(which="major", length=3, color=CELL_LINE_COLOR)
        ax.tick_params(which="minor", length=0)
        for spine in ax.spines.values():
            spine.set_linewidth(CELL_LINE_WIDTH)
        ax.set_xlabel("Message Size [Byte]")

        for row in range(len(collectives)):
            for column in range(len(message_sizes)):
                ax.text(column, row, format_overhead(values[row, column]),
                        ha="center", va="center", color="black", fontsize=10)

    fig.legend(
        handles=[Patch(facecolor=color, edgecolor=CELL_LINE_COLOR,
                       linewidth=CELL_LINE_WIDTH, label=label)
                 for label, color in zip(SEVERITY_LABELS, SEVERITY_COLORS)],
        loc="outside upper center", ncol=len(SEVERITY_LABELS), fontsize=9,
        frameon=True, fancybox=True, edgecolor="#666666",
    )
    if caption:
        fig.supxlabel(caption, fontsize=10)
    if filename and SAVE_OUTPUT:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(OUTPUT_DIR / filename, dpi=200, bbox_inches="tight")
        print("Saved:", OUTPUT_DIR / filename)
    plt.show()


nodes = int(executions["node_count"].iloc[0])
ppn = int(executions["processes_per_node"].iloc[0])
caption = (f"{EXPERIMENT_FILTER}; {nodes * ppn} ({nodes} x {ppn}) processes, "
           f"{overhead['execution'].nunique()} executions")

plot_overhead_panels(
    [("Best", best_cells), ("Worst", worst_cells)],
    caption=caption,
    filename="overhead_best_worst_cellwise.png",
)

# The paper's cells carry no run label, so the origin of each extreme is listed here.
print("Origin of the extreme values:")
display(
    best_cells[[*CELL_KEYS, "overhead_pct", "source"]]
    .merge(worst_cells[[*CELL_KEYS, "overhead_pct", "source"]], on=CELL_KEYS,
           suffixes=("_best", "_worst"))
    .round(1)
)

## 7. Variant: the best vs. the worst execution as a whole

Same layout, but each panel now shows **one** complete execution.

In [ ]:
plot_overhead_panels(
    [(f"Best ({best_execution['label']})",
      overhead[overhead["execution"] == best_execution["execution"]]),
     (f"Worst ({worst_execution['label']})",
      overhead[overhead["execution"] == worst_execution["execution"]])],
    caption=f"{caption}; ranked by the '{RANK_METRIC}' of o",
    filename="overhead_best_worst_execution.png",
)

## 8. Every execution on its own (layout of Fig. 7)

Fig. 7 of the paper reuses the same tile plot, but with **variants as rows**
(`MPI_Scan (def)` vs. `MPI_Scan (alt)`) instead of collectives. Same idea here: one row
per execution. The run-to-run spread is then readable directly, without any min/max
aggregation.

In [ ]:
RUN_ROWS = list(executions["label"])

for collective in COLLECTIVES:
    plot_overhead_panels(
        [(collective, overhead[overhead["collective"] == collective])],
        caption=f"{caption}; one row per execution",
        rows=RUN_ROWS, row_key="label",
        filename=f"overhead_per_execution_{collective}.png",
    )

## 9. Comparing the algorithms (Fig. 6 and Fig. 8)

Two original figures, two views of the same numbers:

* **Fig. 6** -- bar chart of the median runtimes of all implementations at **one**
  message size, value printed above the bar, ColorBrewer palette `Set2`.
* **Fig. 8** -- lines across all message sizes, logarithmic y axis, categorical x axis,
  ColorBrewer palette `Dark2`. The original compares **aggregates**, not every single
  variant: `Default Algorithm (autoselect)`, `Best Tuned Algorithm` and
  `Guideline-Based Algorithm`. Reproduced here with the two curves this experiment has,
  the library default (`eta_L`) and the fastest guideline implementation (`eta_M`); the
  tuned curve would need an exhaustive search over Open MPI's own algorithms, which this
  experiment does not run. The vertical gap between the two curves is exactly the
  overhead `o` that the heatmaps encode.

So the two figures are not the same plot twice: Fig. 6 resolves **every implementation
at one message size**, Fig. 8 resolves **the two aggregates across all message sizes**.
Set `LINE_SERIES = "all_variants"` to get one line per mock-up instead -- useful to see
*which* mock-up wins where, which the per-size `fastest_mockup` column of
`overhead.csv` also records.

The colour values were read out of the PDF's vector paths and identified as ColorBrewer
palettes (`Set2` in Fig. 6, `Dark2` in Fig. 8), which suggests the originals were drawn
with R/ggplot2.

In [ ]:
# ColorBrewer palettes, read out of the PDF: Set2 for the bars, Dark2 for the lines.
SET2 = ["#66c2a5", "#fc8d62", "#8da0cb", "#e78ac3",
        "#a6d854", "#ffd92f", "#e5c494", "#b3b3b3"]
DARK2 = ["#1b9e77", "#d95f02", "#7570b3", "#e7298a",
         "#66a61e", "#e6ab02", "#a6761d", "#666666"]
GRID_GREY = "#c8c8c8"

# Fig. 6 shows a single message size. None = per collective the largest one it
# was measured with (collectives whose buffers scale with N stop earlier).
BAR_MESSAGE_SIZE = None

# "paper" reproduces Fig. 8 (library default vs. fastest guideline implementation),
# "all_variants" draws one line per mock-up instead.
LINE_SERIES = "paper"

# Colors and markers of Fig. 8: default orange circles, guideline green squares.
# The third Dark2 slot stays reserved for the paper's "Best Tuned Algorithm".
FIG8_STYLE = {
    "Default Algorithm (autoselect)": ("#d95f02", "o"),
    "Guideline-Based Algorithm": ("#1b9e77", "s"),
}


def variant_label(algorithm: str) -> str:
    """'bcast_as_bcast_lane' -> 'As Bcast Lane', like the axis labels in Fig. 6."""
    if algorithm == "default":
        return "Default"
    # The mock-up name drops the underscores of the MPI name
    # ('reducescatterblock_as_allreduce'), so everything in front of the first
    # '_as_' is cut instead of matching a prefix built from the collective --
    # otherwise the label repeats it: 'As Reducescatterblock As Allreduce'.
    _, separator, rest = algorithm.partition("_as_")
    if not separator:
        rest = algorithm
    return "As " + rest.replace("_", " ").title()


def plot_runtime_bars(execution, collective, message_size=None, filename=None):
    """Fig. 6: median runtime of every implementation at one message size."""
    available = medians[(medians["collective"] == collective)
                        & (medians["execution"] == execution["execution"])]
    if available.empty:
        return
    if message_size is None:
        message_size = available["message_size"].max()
    subset = available[available["message_size"] == message_size].copy()
    if subset.empty:
        print(f"{collective} @ {format_size(message_size)} B: no data")
        return
    # Fig. 6 keeps a fixed order: Default first, then the mock-ups alphabetically.
    subset["order"] = np.where(subset["algorithm"] == "default", "", subset["algorithm"])
    subset = subset.sort_values("order")

    labels = [variant_label(row.algorithm) for row in subset.itertuples()]
    values = subset["median_ms"].to_numpy()
    fig, ax = plt.subplots(figsize=(0.85 * len(labels) + 2.0, 3.4), layout="constrained")
    bars = ax.bar(labels, values, color=SET2[:len(labels)],
                  edgecolor="black", linewidth=CELL_LINE_WIDTH, width=0.8)
    ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=8)
    ax.set_ylabel("Time (ms)")
    ax.set_ylim(0, values.max() * 1.18)
    ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right")
    ax.grid(axis="y", color=GRID_GREY, linewidth=CELL_LINE_WIDTH)
    ax.set_axisbelow(True)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    ax.set_title(f"{collective}, {format_size(message_size)} B -- {execution['label']}",
                 fontsize=10)
    if filename and SAVE_OUTPUT:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(OUTPUT_DIR / filename, dpi=200, bbox_inches="tight")
    plt.show()


def aggregate_curves(subset: pd.DataFrame) -> dict:
    """The two curves of Fig. 8: the library default and the fastest mock-up per size."""
    library = subset[subset["is_default"]].groupby("message_size")["median_ms"].min()
    mockups = subset[~subset["is_default"]].groupby("message_size")["median_ms"].min()
    return {
        "Default Algorithm (autoselect)": library,
        "Guideline-Based Algorithm": mockups,
    }


def plot_runtime_lines(execution, collective, filename=None):
    """Fig. 8: runtime over all message sizes, log y, categorical x."""
    subset = medians[(medians["collective"] == collective)
                     & (medians["execution"] == execution["execution"])]
    if subset.empty:
        return
    sizes = sorted(subset["message_size"].unique())
    positions = {size: index for index, size in enumerate(sizes)}

    fig, ax = plt.subplots(figsize=(5.2, 3.4), layout="constrained")
    if LINE_SERIES == "paper":
        for label, series in aggregate_curves(subset).items():
            color, marker = FIG8_STYLE[label]
            ax.plot([positions[size] for size in series.index], series.to_numpy(),
                    color=color, marker=marker, markersize=4.5, linewidth=1.2, label=label)
    else:
        algorithms = ["default"] + sorted(a for a in subset["algorithm"].unique()
                                          if a != "default")
        markers = ["o", "^", "s", "D", "v", "P", "X", "*"]
        for index, algorithm in enumerate(algorithms):
            series = subset[subset["algorithm"] == algorithm].sort_values("message_size")
            ax.plot([positions[size] for size in series["message_size"]], series["median_ms"],
                    color=DARK2[index % len(DARK2)], marker=markers[index % len(markers)],
                    markersize=4.5, linewidth=1.2,
                    label=variant_label(algorithm))
    ax.set_yscale("log")
    # Fig. 8 prints plain decimals (0.01, 0.10, 1.00), not powers of ten.
    ax.yaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value:g}"))
    ax.yaxis.set_minor_formatter(NullFormatter())
    ax.set_xticks(range(len(sizes)), [format_size(size) for size in sizes],
                  rotation=45, ha="right")
    ax.set_xlabel("Message Size [Byte]")
    ax.set_ylabel("Execution Time [ms]")
    ax.set_title(f"{collective} -- {execution['label']}", fontsize=10)
    ax.legend(fontsize=7.5, loc="upper left", frameon=True, edgecolor="black",
              framealpha=1.0, borderpad=0.4).get_frame().set_linewidth(CELL_LINE_WIDTH)
    for spine in ax.spines.values():
        spine.set_linewidth(CELL_LINE_WIDTH)
    if filename and SAVE_OUTPUT:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(OUTPUT_DIR / filename, dpi=200, bbox_inches="tight")
    plt.show()


panel_runs = [("Best", best_execution), ("Worst", worst_execution)]
for collective in COLLECTIVES:
    for role, execution in panel_runs:
        plot_runtime_bars(execution, collective, BAR_MESSAGE_SIZE,
                          filename=f"runtime_bars_{collective}_{role.lower()}.png")
        plot_runtime_lines(execution, collective,
                           filename=f"runtime_lines_{collective}_{role.lower()}.png")

for role, execution in panel_runs:
    print(f"\n{role}: {execution['label']} -- median runtime [ms]")
    display(
        medians[medians["execution"] == execution["execution"]]
        .pivot_table(index=["collective", "message_size"], columns="algorithm",
                     values="median_ms")
        .round(4)
    )

## 10. The tables from the paper

* **Table 1** -- hardware overview: nodes, CPU, memory, interconnect, MPI version. All of
  it read from the `allocation.json` and `run.log` of the runs, nothing entered by hand.
* **Table 2** -- share of measurements with a *high* or *very high* violation
  (`o > 0.5`) per collective, one column per machine, lower is better. The denominator
  is every single measurement, i.e. message sizes times executions -- not one run. That
  is verifiable against the paper: Leonardo/`MPI_Bcast` is 38 %, which is 8 of 21
  (7 sizes x 3 batch jobs); the best run alone would give 29 %, the worst alone 43 %.
  A second table breaks the same numbers down per experiment, which the paper does not
  show.

Both tables are also written as LaTeX (`booktabs`) so they can be pasted into the thesis
directly.

In [ ]:
NODE_PATTERNS = {
    "node_type": r"type='([^']+)'",
    "cpu": r"'model': '([^']*(?:Xeon|EPYC|Ryzen)[^']*)'",
    "ram": r"'humanized_ram_size': '([^']+)'",
    "cores": r"'smt_size': (\d+)",
    "sockets": r"'smp_size': (\d+)",
    "nic": r"'model': '(MT\d+[^']*)'",
    "nic_rate": r"'rate': (\d+)",
    "site": r"site='([^']+)'",
}


def node_facts(execution_path: Path) -> dict:
    """Hardware of one allocation, parsed out of the Node repr in allocation.json."""
    allocation = load_json(execution_path.parent / "allocation.json")
    nodes = allocation.get("selected_nodes") or []
    if not nodes:
        return {}
    raw = nodes[0].get("raw") or ""
    facts = {}
    for key, pattern in NODE_PATTERNS.items():
        match = re.search(pattern, raw)
        facts[key] = match.group(1) if match else None
    facts["machines"] = ", ".join(node.get("name", "?") for node in nodes)
    facts["node_count"] = len(nodes)
    return facts


def mpi_version(execution_path: Path) -> str:
    """OpenMPI version from the libmpi path that run.sh logs per node."""
    log = execution_path / "run.log"
    if not log.exists():
        return "?"
    match = re.search(r"openmpi-([0-9][0-9.]*)", log.read_text(errors="replace"))
    return f"Open MPI {match.group(1)}" if match else "?"


rows = []
for row in executions.itertuples(index=False):
    facts = node_facts(row.path)
    if not facts:
        continue
    rate = facts.get("nic_rate")
    rows.append({
        "Experiment": row.label,
        "Site": facts.get("site"),
        "Compute nodes": f"{facts['node_count']} x {facts.get('node_type')}",
        "CPU": (f"{facts.get('sockets')} x {facts.get('cpu')}"
                if facts.get("cpu") else None),
        "Cores/node": facts.get("cores"),
        "Memory": facts.get("ram"),
        "Interconnect": (f"{facts.get('nic')}"
                         + (f", {int(rate) // 10**9} GbE" if rate else "")),
        "MPI library": mpi_version(row.path),
        "Machines": facts.get("machines"),
    })

table1 = pd.DataFrame(rows)
if table1.empty:
    print("Table 1 -- skipped: no selected_nodes in allocation.json")
else:
    # Allocations with identical hardware collapse into one row.
    hardware_columns = [column for column in table1.columns if column != "Experiment"]
    table1 = (
        table1.groupby(hardware_columns, dropna=False)["Experiment"]
        .agg(lambda labels: ", ".join(sorted(labels)))
        .reset_index()[["Experiment", *hardware_columns]]
    )
    print("Table 1 -- hardware")
    display(table1)

# Table 2: share of measurements above the "high" threshold (o > 0.5). The paper
# pools all batch jobs of a machine, so the denominator is message sizes x executions.
HIGH_THRESHOLD = 0.5


def as_percent(frame: pd.DataFrame, index_name: str) -> pd.DataFrame:
    formatted = frame.mul(100).round(0).astype(int).astype(str) + " %"
    formatted.index.name = index_name
    formatted.columns.name = None
    return formatted


violated = overhead.assign(violated=overhead["overhead"] > HIGH_THRESHOLD)
violated["experiment_short"] = violated["label"].str.slice(0, 1)

# One column per machine, as in the paper. `machines` is the node set of the
# allocation, so runs on identical hardware land in the same column.
machine_names = {}
for row in executions.itertuples(index=False):
    facts = node_facts(row.path)
    machine_names[row.label] = (f"{facts.get('site')} / {facts.get('node_type')}"
                                if facts else "unknown machine")
violated["machine"] = violated["label"].map(machine_names)

table2 = violated.pivot_table(index="collective", columns="machine",
                              values="violated", aggfunc="mean")
table2["Total"] = violated.groupby("collective")["violated"].mean()

print(f"\nTable 2 -- share of measurements with o > {HIGH_THRESHOLD} [%], lower is better")
print(f"Denominator per cell: message sizes x executions "
      f"({violated.groupby('collective').size().to_dict()} measurements)")
table2_display = as_percent(table2, "MPI call")
display(table2_display)

table2_by_experiment = as_percent(
    violated.pivot_table(index="collective", columns="experiment_short",
                         values="violated", aggfunc="mean"),
    "MPI call")
print("\nSame numbers per experiment (not in the paper):")
display(table2_by_experiment)

if SAVE_OUTPUT:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    # escape=True is required: pandas 2.x does not escape by default, and an
    # unescaped MPI_Bcast or "%" would break the LaTeX build.
    if not table1.empty:
        (OUTPUT_DIR / "table1_hardware.tex").write_text(
            table1.to_latex(index=False, escape=True), encoding="utf-8")
        table1.to_csv(OUTPUT_DIR / "table1_hardware.csv", index=False)
    (OUTPUT_DIR / "table2_violations.tex").write_text(
        table2_display.to_latex(escape=True), encoding="utf-8")
    table2_display.to_csv(OUTPUT_DIR / "table2_violations.csv")
    table2_by_experiment.to_csv(OUTPUT_DIR / "table2_violations_by_experiment.csv")
    print("\nLaTeX/CSV written:", OUTPUT_DIR / "table1_hardware.tex",
          "|", OUTPUT_DIR / "table2_violations.tex")